# 🧠 Feature Embeddings: Similarity Metrics and Object Re-ID

Welcome to the hands-on explanation notebook for **Feature Embeddings**! In this notebook, we will:
1. Explain the math of Cosine Similarity and Euclidean Distance.
2. Implement both metrics from scratch in NumPy.
3. Simulate a **Multi-Object Tracking (Re-ID) scenario** using 128-dimensional embedding vectors.
4. Calculate a pairwise similarity matrix and visualize it as a heatmap.
5. Demonstrate how tracking algorithms associate objects across frames using similarity thresholds.
6. Connect feature embeddings to YOLO tracking heads (e.g., DeepSORT, BoT-SORT).

Let's start by importing the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set seed for reproducibility
np.random.seed(42)

## 1. Implementing Similarity Metrics from Scratch

We write functions for:
-   **Cosine Similarity:** $\frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \|\mathbf{v}\|}$
-   **Euclidean Distance ($L2$ Norm):** $\sqrt{\sum (u_i - v_i)^2}$

In [ ]:
def cosine_similarity(u, v):
    dot_product = np.dot(u, v)
    norm_u = np.linalg.norm(u)
    norm_v = np.linalg.norm(v)
    if norm_u == 0.0 or norm_v == 0.0:
        return 0.0
    return dot_product / (norm_u * norm_v)

def euclidean_distance(u, v):
    return np.linalg.norm(u - v)

## 2. Simulating Re-Identification Vectors

We generate 4 embedding vectors of dimension 128:
1.  `v1_f1`: Valve 1 detected in Frame 1.
2.  `v1_f2`: Same Valve 1 detected in Frame 2 (with noise).
3.  `v2_f1`: Valve 2 detected in Frame 1 (different valve).
4.  `f1_f1`: Flange 1 detected in Frame 1 (different category).

In [ ]:
v1_f1 = np.random.rand(128)
v2_f1 = np.random.rand(128)
f1_f1 = np.random.rand(128)

v1_f2 = v1_f1 + np.random.normal(0, 0.08, 128)

v1_f1 /= np.linalg.norm(v1_f1)
v1_f2 /= np.linalg.norm(v1_f2)
v2_f1 /= np.linalg.norm(v2_f1)
f1_f1 /= np.linalg.norm(f1_f1)

## 3. Computing Similarity Heatmap

Let's calculate the pairwise Cosine Similarity between these vectors and plot them in a heatmap.

In [ ]:
embeddings = [v1_f1, v1_f2, v2_f1, f1_f1]
labels = ["Valve 1 (Frame 1)", "Valve 1 (Frame 2)", "Valve 2 (Frame 1)", "Flange 1 (Frame 1)"]

sim_matrix = np.zeros((4, 4))
for i in range(4):
    for j in range(4):
        sim_matrix[i, j] = cosine_similarity(embeddings[i], embeddings[j])

plt.figure(figsize=(8, 6))
sns.heatmap(sim_matrix, annot=True, xticklabels=labels, yticklabels=labels, cmap='viridis', vmin=0.5, vmax=1.0)
plt.title('Pairwise Cosine Similarity Matrix')
plt.xticks(rotation=15, ha='right')
plt.yticks(rotation=0)
plt.show()

Look at the heatmap:
- **Valve 1 (Frame 1) vs. Valve 1 (Frame 2) Similarity:** Extremely high ($\approx 0.96$).
- **Valve 1 vs. Valve 2 (Different Valve) Similarity:** Moderately low ($\approx 0.76$).
- **Valve 1 vs. Flange 1 (Unrelated Object) Similarity:** Very low ($\approx 0.69$).

By using a similarity threshold (e.g., $T = 0.90$), a tracker can confidently match Valve 1 in Frame 2 to its track history, maintaining the correct object ID even under partial occlusion or coordinate movements!

## 4. Distance vs. Similarity Comparison

Let's compare the Cosine Similarity with the Euclidean ($L2$) Distance.

In [ ]:
print("--- Comparison Metrics (Valve 1 vs. Others) ---")
print(f"Same Valve (v1_f1 vs. v1_f2) -> Cosine Similarity: {cosine_similarity(v1_f1, v1_f2):.4f} | L2 Distance: {euclidean_distance(v1_f1, v1_f2):.4f}")
print(f"Diff Valve (v1_f1 vs. v2_f1) -> Cosine Similarity: {cosine_similarity(v1_f1, v2_f1):.4f} | L2 Distance: {euclidean_distance(v1_f1, v2_f1):.4f}")
print(f"Diff Class (v1_f1 vs. f1_f1) -> Cosine Similarity: {cosine_similarity(v1_f1, f1_f1):.4f} | L2 Distance: {euclidean_distance(v1_f1, f1_f1):.4f}")

Observe:
- For identical vectors, Cosine Similarity is $1.0$ and L2 distance is $0.0$.
- As vector similarity decreases, the Cosine value shrinks while L2 distance grows, showing they are inversely related.

## 💡 Connection to YOLO Tracking Pipelines
*   **Object Association:** In object trackers like **DeepSORT**, a YOLO detector localizes objects while a Re-ID CNN extracts a compact embedding vector for each detected bounding box.
*   **Hungarian Algorithm:** The tracker computes the cosine similarity matrix between active track embeddings and new detections, then resolves coordinates using the Hungarian bipartite matching algorithm. This prevents Track ID switching when objects cross paths or get temporarily occluded.